# ATLAS — Multimodal Training

Boilerplate for feature-level late fusion. **All inputs are cleaned, preprocessed, and null-handled.** Fill in the TODO cells.

| Component | Owner | Output |
| --- | --- | --- |
| S3 image tensors → ResNet-18 | Kevin | `[B, 512]` |
| Tabular features → MLP | Swati | `[B, 64]` |
| USGS DEM → SmallCNN | Abhi | `[B, 64]` |
| Fusion, training, evaluation, XAI | Team — owner TBD | `[B, 640]` → logits |

`B` = batch size. Concatenation order: **image, tabular, DEM**.

## 1. Setup
**Owner: Team — TBD**

Set imports, device, seed, S3 paths, class names, and training hyperparameters.

In [ ]:

# Section 1 — Shared multimodal configuration
import io
import random
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
N_CLASSES = 3
CLASS_NAMES = ("Small", "Medium", "Large")
N_FOLDS = 5

IMAGE_EMB_DIM = 512
TAB_EMB_DIM = 64
DEM_EMB_DIM = 64
FUSION_DIM = IMAGE_EMB_DIM + TAB_EMB_DIM + DEM_EMB_DIM

BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

AWS_PROFILE = "ATLAS"
AWS_REGION = "us-east-2"
S3_BUCKET = "atlas-acm-research-f26"

IMAGE_PREFIX = "satellite/processed/"
DEM_PREFIX = "usgs/naip_aligned_v2/dem/"

TABULAR_KEY = (
    "dataset/atlas_wri_13score_v1/"
    "im3_atlas_wri_13scores_normalized_0to1.csv"
)

SPATIAL_FOLDS_KEY = (
    "artifacts/usgs_terrain_baselines/folds/v1/"
    "usgs_spatial_folds_v1.csv"
)

DEM_CHECKPOINT_PREFIX = (
    "artifacts/usgs_terrain_baselines/multimodal_compatible/"
    "20261104-053909-566752/cnn/"
)

# Resolve the repository root whether the notebook runs
# from ATLAS/ or ATLAS/models/.
cwd = Path.cwd().resolve()

if (cwd / "models" / "train_multimodal.ipynb").is_file():
    PROJECT_ROOT = cwd
elif (cwd / "train_multimodal.ipynb").is_file() and cwd.name == "models":
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        f"Cannot locate ATLAS repository root from {cwd}"
    )

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "multimodal_spatial_v1"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

s3_shared = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=AWS_REGION
).client("s3")

print("Device:", device)
print("Fusion dimension:", FUSION_DIM)
print("S3 bucket:", S3_BUCKET)
print("Spatial folds:", N_FOLDS)


Device: cuda
Fusion dimension: 640
S3 bucket: atlas-acm-research-f26
Spatial folds: 5


## 2. Load aligned inputs and split assignments
**Owner: Team — TBD**

Load the prepared facility manifest and shared train/validation/test assignments. Use facility IDs to retrieve matching image, tabular, DEM, and target inputs.

In [4]:

# Section 2 — Shared facility manifest and spatial folds

def read_shared_s3_csv(key):
    response = s3_shared.get_object(
        Bucket=S3_BUCKET,
        Key=key
    )
    return pd.read_csv(
        io.BytesIO(response["Body"].read()),
        dtype={"id": str}
    )


def normalize_facility_ids(series):
    ids = series.astype("string").str.strip()
    assert ids.notna().all(), "Missing facility IDs"
    assert ids.str.fullmatch(r"\d{1,10}").all(), (
        "Facility IDs must contain 1–10 digits"
    )
    return ids.str.zfill(10)


spatial_folds = read_shared_s3_csv(SPATIAL_FOLDS_KEY)
tabular_source = read_shared_s3_csv(TABULAR_KEY)

required_fold_columns = {
    "id", "lat", "lon", "size_class", "geo_group", "fold"
}
assert required_fold_columns.issubset(spatial_folds.columns)

required_tabular_columns = {"id", "size_class"}
assert required_tabular_columns.issubset(tabular_source.columns)

spatial_folds["id"] = normalize_facility_ids(
    spatial_folds["id"]
)
tabular_source["id"] = normalize_facility_ids(
    tabular_source["id"]
)

assert spatial_folds["id"].is_unique
assert tabular_source["id"].is_unique
assert len(spatial_folds) == 180, (
    f"Expected 180 facilities, got {len(spatial_folds)}"
)

assert set(spatial_folds["fold"]) == set(range(1, N_FOLDS + 1))

manifest = spatial_folds[
    ["id", "lat", "lon", "size_class", "geo_group", "fold"]
].copy()

tab_labels = tabular_source[
    ["id", "size_class"]
].rename(columns={"size_class": "tabular_size_class"})

manifest = manifest.merge(
    tab_labels,
    on="id",
    how="left",
    validate="one_to_one"
)

assert manifest["tabular_size_class"].notna().all(), (
    "Some facilities are missing from tabular data"
)

assert (
    pd.to_numeric(manifest["size_class"])
    == pd.to_numeric(manifest["tabular_size_class"])
).all(), "Spatial-fold and tabular labels disagree"

manifest = manifest.drop(columns=["tabular_size_class"])

manifest["image_key"] = (
    IMAGE_PREFIX + manifest["id"] + ".pt"
)
manifest["dem_key"] = (
    DEM_PREFIX + manifest["id"] + "_dem.tif"
)

assert manifest["id"].is_unique
assert manifest["size_class"].notna().all()
assert manifest["geo_group"].notna().all()

manifest["size_class"] = pd.to_numeric(
    manifest["size_class"]
).astype(int)

assert manifest["size_class"].isin(
    range(N_CLASSES)
).all()

assert (
    manifest.groupby("geo_group")["fold"].nunique() == 1
).all(), "Geographic group appears in multiple folds"

manifest = manifest.sort_values("id").reset_index(drop=True)

# Section 6 will use this shared dataframe.
MULTIMODAL_MANIFEST = manifest

print("Multimodal manifest:", MULTIMODAL_MANIFEST.shape)
print("Unique facilities:", MULTIMODAL_MANIFEST["id"].nunique())
print("Spatial fold counts:")
print(MULTIMODAL_MANIFEST["fold"].value_counts().sort_index())
print("Class distribution:")
print(MULTIMODAL_MANIFEST["size_class"].value_counts().sort_index())


Multimodal manifest: (180, 8)
Unique facilities: 180
Spatial fold counts:
fold
1    36
2    37
3    36
4    35
5    36
Name: count, dtype: int64
Class distribution:
size_class
0      7
1     50
2    123
Name: count, dtype: int64


# temp check section

In [5]:
fold_class_counts = pd.crosstab(
    MULTIMODAL_MANIFEST["fold"],
    MULTIMODAL_MANIFEST["size_class"]
)

fold_class_counts = fold_class_counts.reindex(
    index=range(1, N_FOLDS + 1),
    columns=range(N_CLASSES),
    fill_value=0,
)

fold_class_counts.columns = CLASS_NAMES

print("Class distribution by geographic fold:")
print(fold_class_counts)

print("\nFolds missing Small facilities:")
print(fold_class_counts.index[
    fold_class_counts["Small"] == 0
].tolist())

Class distribution by geographic fold:
      Small  Medium  Large
fold                      
1         1      10     25
2         2      10     25
3         1      10     25
4         1      10     24
5         2      10     24

Folds missing Small facilities:
[]


In [6]:

# Section 2b — Rotating geographic train/val/test splits

def make_spatial_split(manifest, test_fold):
    """Use one outer test fold, the next fold for validation."""
    if test_fold not in range(1, N_FOLDS + 1):
        raise ValueError(f"Invalid test fold: {test_fold}")

    val_fold = (test_fold % N_FOLDS) + 1

    result = manifest.copy()
    result["split"] = "train"
    result.loc[result["fold"] == val_fold, "split"] = "val"
    result.loc[result["fold"] == test_fold, "split"] = "test"

    assert result["id"].is_unique
    assert set(result["split"]) == {"train", "val", "test"}

    # No geographic group may cross a split boundary.
    assert (
        result.groupby("geo_group")["split"].nunique() == 1
    ).all(), "Geographic leakage detected"

    for split in ("train", "val", "test"):
        subset = result.loc[result["split"] == split]
        assert not subset.empty, f"{split} partition is empty"

    return result


# Initial integration run: fold 1 held out for testing.
TEST_FOLD = 1
MULTIMODAL_SPLIT = make_spatial_split(
    MULTIMODAL_MANIFEST,
    test_fold=TEST_FOLD
)

print("\nInitial geographic split:")
print(
    pd.crosstab(
        MULTIMODAL_SPLIT["split"],
        MULTIMODAL_SPLIT["size_class"]
    ).reindex(["train", "val", "test"])
)

print("\nPartition sizes:")
print(
    MULTIMODAL_SPLIT["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)



Initial geographic split:
size_class  0   1   2
split                
train       4  30  73
val         2  10  25
test        1  10  25

Partition sizes:
split
train    107
val       37
test      36
Name: count, dtype: int64


## 3. Image encoder — Kevin

Load prepared image tensors from S3 (`satellite/processed/`). Each `.pt` contains `id`, `image`, and `source_key`.

**`[B, 3, 224, 224]` → ResNet-18 → `[B, 512]`**

Extract features before the final classification layer. Specify checkpoint and frozen/trainable layers.

In [12]:
from io import BytesIO
import boto3
import torch
from torch import nn
from torchvision import models, transforms

BUCKET="atlas-acm-research-f26"
IMAGE_PREFIX="satellite/processed/"
#imagenet weights
IMAGE_WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1
IMAGE_CHECKPOINT_KEY = None
IMAGE_FREEZE_BACKBONE = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
s3 = boto3.Session(
    profile_name="ATLAS",
    region_name="us-east-2",
).client("s3")

image_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

def read_s3_bytes(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
    try:
        return body.read()
    finally:
        body.close()
        
def load_image_tensor(s3_key):
    payload = torch.load(
        BytesIO(read_s3_bytes(s3_key)),
        map_location="cpu",
        weights_only=True,
    )
    image = payload["image"].float()

    if image.shape != (3, 224, 224):
        raise ValueError(
            f"Unexpected image shape for {s3_key}: {tuple(image.shape)}"
        )

    #stored tensors are RGB values in [0, 1].
    #apply normalization once, matching the visual baseline.
    return image_normalize(image)

class ImageEncoder(nn.Module):
    output_dim = 512

    def __init__(self, checkpoint_key=None, freeze_backbone=True):
        super().__init__()

        #custom ATLAS checkpoints require training-ID and fold-compatibility validation before loading.
        if checkpoint_key is not None:
            raise ValueError(
                "ATLAS vision checkpoints require training-ID and "
                "fold-compatibility validation before loading."
            )
        weights = IMAGE_WEIGHTS
        
        self.backbone = models.resnet18(weights=weights)


        #return image features instead of Small/Medium/Large classification
        self.backbone.fc = nn.Identity()

        self.freeze_backbone = freeze_backbone
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not freeze_backbone

        if freeze_backbone:
            self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)

        #preserve frozen BatchNorm statistics during fused training.
        if self.freeze_backbone:
            self.backbone.eval()

        return self

    def forward(self, images):
        #input: normalized [B, 3, 224, 224]
        #output: [B, 512]
        return self.backbone(images)


IMAGE_ENCODER_CONFIG = {
    "architecture": "resnet18",
    "initialization": IMAGE_WEIGHTS.name,
    "atlas_checkpoint_key": IMAGE_CHECKPOINT_KEY,
    "freeze_backbone": IMAGE_FREEZE_BACKBONE,
    "batchnorm": (
        "fixed running statistics"
        if IMAGE_FREEZE_BACKBONE
        else "updated during training"
    ),
    "output_dim": 512,
}

image_encoder = ImageEncoder(
    checkpoint_key=IMAGE_CHECKPOINT_KEY,
    freeze_backbone=IMAGE_FREEZE_BACKBONE,
).to(device)

print("Image encoder configuration:")
for name, value in IMAGE_ENCODER_CONFIG.items():
    print(f"  {name}: {value}")

print("Device:", device)
print(
    "Trainable parameters:",
    sum(p.numel() for p in image_encoder.parameters() if p.requires_grad),
) 

Image encoder configuration:
  architecture: resnet18
  initialization: IMAGENET1K_V1
  atlas_checkpoint_key: None
  freeze_backbone: True
  batchnorm: fixed running statistics
  output_dim: 512
Device: cuda
Trainable parameters: 0


## 4. Tabular encoder — Swati

Load prepared IM3 + EIA + WRI features and adapt `TabularEncoder` for fusion.

**`[B, F]` → MLP → `[B, 64]`**

`F` is the prepared feature count. Return embeddings; train with the fused classification loss.

In [ ]:
# TODO (Swati): Load prepared tabular features and define the MLP encoder.
# [B, F] -> [B, 64]

In [8]:
# Tabular config + load — shared multimodal source
# Uses Section 2's verified S3 tabular data and facility IDs.

TAB_EMB_DIM = 64

TAB_ID_COLS = ["id", "name", "ref", "operator", "county", "state", "state_abb", "state_id", "county_id",
               "county_fips", "pfaf_id", "retail_period", "period"]
TAB_LOCATION_COLS = ["lat", "lon"]          # so the model can't just memorize location
TAB_LEAKAGE_COLS = ["sqft", "size_class"]   # size_class is built from sqft -> leaks the size answer
TAB_DROP_COLS = TAB_ID_COLS + TAB_LOCATION_COLS + TAB_LEAKAGE_COLS

# Restrict tabular records to facilities with aligned modalities.
shared_ids = MULTIMODAL_MANIFEST["id"].tolist()

tab_df = tabular_source.loc[
    tabular_source["id"].isin(shared_ids)
].copy()

assert tab_df["id"].is_unique
assert len(tab_df) == len(shared_ids) 
assert set(tab_df["id"]) == set(shared_ids)

# Preserve manifest ordering.
tab_df = (
    tab_df.set_index("id")
    .loc[shared_ids]
    .reset_index()
)

assert tab_df["id"].tolist() == shared_ids

print("Aligned tabular records:", tab_df.shape)


Aligned tabular records: (180, 63)


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def prepare_tabular_fold(aligned_tabular, split_df):
    #work on copies so preparing another fold cannot overwrite this one.
    frame = aligned_tabular.drop(columns=["split"], errors="ignore").copy()

    assignments = split_df[["id", "split", "size_class"]].rename(
        columns={"size_class": "shared_size_class"}
    )

    assert frame["id"].is_unique
    assert assignments["id"].is_unique
    assert set(frame["id"]) == set(assignments["id"])

    frame = frame.merge(
        assignments,
        on="id",
        how="left",
        validate="one_to_one",
        sort=False,
    )

    assert frame["split"].notna().all()
    assert (
        pd.to_numeric(frame["size_class"])
        == pd.to_numeric(frame["shared_size_class"])
    ).all(), "Tabular and shared labels disagree"

    frame = frame.drop(columns="shared_size_class")

    ids_by_split = {
        name: frame.loc[frame["split"] == name, "id"].tolist()
        for name in ("train", "val", "test")
    }

    for name, ids in ids_by_split.items():
        assert ids, f"{name} partition is empty"

    train_ids = set(ids_by_split["train"])
    val_ids = set(ids_by_split["val"])
    test_ids = set(ids_by_split["test"])

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    excluded = TAB_DROP_COLS + ["split", "fold", "geo_group"]
    features = frame.drop(
        columns=[column for column in excluded if column in frame]
    )
    train_mask = frame["split"].eq("train")

    #feature selection is learned from this fold's training rows only.
    constant_columns = [
        column
        for column in features
        if features.loc[train_mask, column].nunique(dropna=False) <= 1
    ]
    features = features.drop(columns=constant_columns)

    if features.shape[1] == 0:
        raise ValueError("No usable tabular features remain.")

    numeric_columns = features.select_dtypes(
        include=np.number
    ).columns.tolist()
    categorical_columns = [
        column for column in features if column not in numeric_columns
    ]

    #give sklearn a consistent representation of categorical missing values.
    for column in categorical_columns:
        values = features[column].astype(object)
        features[column] = values.where(values.notna(), np.nan)

    # A NEW, unfitted pipeline is constructed on every function call.
    preprocessor = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            numeric_columns,
        ),
        (
            "cat",
            Pipeline([
                (
                    "impute",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical_columns,
        ),
    ])

    preprocessor.fit(features.loc[train_mask])
    values = preprocessor.transform(features).astype(np.float32)

    assert np.isfinite(values).all(), "Nonfinite tabular features remain"

    # This lookup captures THIS fold's matrix, not a global tab_X variable.
    row_of = {
        facility_id: index
        for index, facility_id in enumerate(frame["id"])
    }

    def lookup(ids):
        rows = [row_of[facility_id] for facility_id in ids]
        return torch.from_numpy(values[rows])

    return {
        "preprocessor": preprocessor,
        "lookup": lookup,
        "input_dim": values.shape[1],
        "feature_names": list(preprocessor.get_feature_names_out()),
        "dropped_constant": constant_columns,
        "ids_by_split": ids_by_split,
        "fit_ids": tuple(ids_by_split["train"]),
    }

In [10]:
#independently prepare tabular preprocessing for every geographic fold.
TABULAR_FOLDS = {}

for test_fold in range(1, N_FOLDS + 1):
    split_df = make_spatial_split(
        MULTIMODAL_MANIFEST,
        test_fold=test_fold,
    )

    context = prepare_tabular_fold(tab_df, split_df)

    expected_train_ids = set(
        split_df.loc[split_df["split"] == "train", "id"]
    )
    held_out_ids = set(
        split_df.loc[split_df["split"].isin(["val", "test"]), "id"]
    )

    #verify this preprocessing belongs to this fold's training facilities.
    assert set(context["fit_ids"]) == expected_train_ids
    assert set(context["fit_ids"]).isdisjoint(held_out_ids)

    #each fold must own a separate fitted pipeline.
    for previous in TABULAR_FOLDS.values():
        assert (
            context["preprocessor"]
            is not previous["tabular"]["preprocessor"]
        )

    TABULAR_FOLDS[test_fold] = {
        "split_df": split_df,
        "tabular": context,
    }

    counts = context["ids_by_split"]
    print(
        f"Fold {test_fold}: "
        f"train={len(counts['train'])}, "
        f"val={len(counts['val'])}, "
        f"test={len(counts['test'])}, "
        f"features={context['input_dim']} | checks passed"
    )

#select the initial fold for the existing notebook checks.
current_fold = TABULAR_FOLDS[TEST_FOLD]
MULTIMODAL_SPLIT = current_fold["split_df"]
tabular_fold = current_fold["tabular"]

get_tabular = tabular_fold["lookup"]
TAB_IN_DIM = tabular_fold["input_dim"]
TAB_FEATURE_NAMES = tabular_fold["feature_names"]

tab_train_ids = tabular_fold["ids_by_split"]["train"]
tab_val_ids = tabular_fold["ids_by_split"]["val"]
tab_test_ids = tabular_fold["ids_by_split"]["test"]

Fold 1: train=107, val=37, test=36, features=69 | checks passed
Fold 2: train=107, val=36, test=37, features=67 | checks passed
Fold 3: train=109, val=35, test=36, features=62 | checks passed
Fold 4: train=109, val=36, test=35, features=65 | checks passed
Fold 5: train=108, val=36, test=36, features=67 | checks passed


In [11]:
# 4d. Tabular MLP encoder: [B, F] -> [B, 64]. Returns embeddings only (no head);
class TabularEncoder(nn.Module):
    def __init__(self, in_dim, emb_dim=TAB_EMB_DIM, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, emb_dim), nn.LayerNorm(emb_dim),
        )
    def forward(self, x):
        return self.net(x)

# shape check on a few training facilities
_enc = TabularEncoder(TAB_IN_DIM).eval()
with torch.no_grad():
    _out = _enc(get_tabular(tab_train_ids[:8]))
assert _out.shape == (8, TAB_EMB_DIM), _out.shape
print(f"TabularEncoder OK: [8, {TAB_IN_DIM}] -> {list(_out.shape)}")
del _enc, _out

TabularEncoder OK: [8, 69] -> [8, 64]


## 5. DEM encoder — Abhi

Load prepared USGS DEM tensors and implement SmallCNN.

**`[B, C_dem, H_dem, W_dem]` → SmallCNN → `[B, 64]`**

Return embeddings before any branch classification head.

In [15]:

# USGS DEM encoder — Abhi
# Pretrained spatial-fold CNN; output: [B, 64].
# No checkpoint is selected automatically: splits must be aligned first.

from pathlib import Path
import numpy as np
import rasterio
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset

DEM_TARGET_SHAPE = (64, 96)


class TerrainCNN(nn.Module):
    def __init__(self, embedding_dim=64, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.embedding = nn.Sequential(
            nn.Linear(64, embedding_dim),
            nn.ReLU()
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def encode(self, image, mask):
        if image.ndim != 4 or image.shape[1] != 1:
            raise ValueError("Expected DEM tensor [B, 1, H, W]")
        if mask.shape != image.shape:
            raise ValueError("DEM and mask shapes must match")

        x = self.features(image * mask)
        m = F.adaptive_avg_pool2d(mask, x.shape[-2:])
        pooled = (
            (x * m).sum(dim=(2, 3))
            / m.sum(dim=(2, 3)).clamp_min(1e-6)
        )
        return self.embedding(pooled)

    def forward(self, image, mask):
        return self.classifier(self.encode(image, mask))


class USGSDemDataset(Dataset):
    """DEM-only dataset, aligned by facility ID.

    Input dataframe requires an 'id' column.
    Returns image, mask, and id; labels remain owned by
    the shared multimodal dataset.
    """
    def __init__(self, dataframe, dem_dir, scale,
                 target_shape=DEM_TARGET_SHAPE):
        self.df = dataframe.reset_index(drop=True).copy()
        self.dem_dir = Path(dem_dir)
        self.scale = float(scale)
        self.target_shape = tuple(target_shape)

        if "id" not in self.df.columns:
            raise ValueError("Dataframe requires an id column")
        if not np.isfinite(self.scale) or self.scale <= 0:
            raise ValueError("scale must be positive and finite")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        facility_id = str(self.df.iloc[idx]["id"])
        path = self.dem_dir / f"{facility_id}_dem.tif"

        with rasterio.open(path) as src:
            dem = src.read(1, masked=True)

        values = dem.compressed()
        if not len(values) or not np.isfinite(values).all():
            raise ValueError(f"Invalid DEM: {facility_id}")

        h, w = dem.shape
        H, W = self.target_shape
        if h > H or w > W:
            raise ValueError(
                f"{facility_id}: {h}x{w} exceeds {H}x{W}"
            )

        mean = float(values.mean())
        centered = np.asarray(
            (dem.filled(mean) - mean) / self.scale,
            dtype=np.float32
        )

        image = np.zeros((H, W), dtype=np.float32)
        mask = np.zeros((H, W), dtype=np.float32)
        top, left = (H - h) // 2, (W - w) // 2

        image[top:top+h, left:left+w] = centered
        mask[top:top+h, left:left+w] = (
            ~np.ma.getmaskarray(dem)
        ).astype(np.float32)

        return {
            "image": torch.from_numpy(image[None]),
            "mask": torch.from_numpy(mask[None]),
            "id": facility_id
        }


def fit_dem_scale(train_df, dem_dir):
    """Fit normalization on training facilities only."""
    sums, count = 0.0, 0
    for fid in train_df["id"]:
        with rasterio.open(
            Path(dem_dir) / f"{fid}_dem.tif"
        ) as src:
            dem = src.read(1, masked=True)

        values = dem.compressed().astype(np.float64)
        if not len(values) or not np.isfinite(values).all():
            raise ValueError(f"Invalid DEM: {fid}")

        sums += np.square(values - values.mean()).sum()
        count += len(values)

    if count == 0:
        raise ValueError("No valid training DEM pixels")

    scale = float(np.sqrt(sums / count))
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid fitted DEM scale")
    return scale


# Explicit DEM configuration:
# pretrained encoder, frozen weights and BatchNorm statistics.
DEM_FREEZE_ENCODER = True


class TerrainFeatureEncoder(nn.Module):
    output_dim = 64

    def __init__(self, terrain, freeze=True):
        super().__init__()
        self.terrain = terrain
        self.freeze_encoder = freeze

        # The standalone DEM classification head is unused by fusion.
        # Remove it after loading the original checkpoint strictly.
        self.terrain.classifier = nn.Identity()

        for parameter in self.terrain.parameters():
            parameter.requires_grad_(not freeze)

        if freeze:
            self.terrain.eval()

    def train(self, mode=True):
        super().train(mode)

        # Fusion model.train() must not update frozen BatchNorm statistics.
        if self.freeze_encoder:
            self.terrain.eval()

        return self

    def encode(self, image, mask):
        return self.terrain.encode(image, mask)

    def forward(self, image, mask):
        return self.encode(image, mask)


def load_terrain_encoder(test_fold, device="cpu", freeze=True):
    """Load only an audited checkpoint compatible with the current split."""
    if test_fold not in VERIFIED_DEM_CHECKPOINTS:
        raise ValueError(
            f"DEM checkpoint for fold {test_fold} is not verified. "
            "Obtain actual checkpoint provenance or a compatible "
            "replacement checkpoint before loading."
        )

    checkpoint = VERIFIED_DEM_CHECKPOINTS[test_fold]
    split_df = TABULAR_FOLDS[test_fold]["split_df"]

    if int(checkpoint["fold"]) != test_fold:
        raise ValueError("DEM checkpoint fold mismatch.")

    # Recheck against current assignments in case splits changed
    # after the audit was executed.
    held_out_ids = set(
        split_df.loc[
            split_df["split"].isin(["val", "test"]),
            "id",
        ]
    )

    for field in PROVENANCE_FIELDS:
        if field not in checkpoint:
            raise ValueError(f"Missing checkpoint provenance: {field}")

        used_ids = checkpoint_id_set(checkpoint[field])

        if field in ("train_ids", "normalization_fit_ids") and not used_ids:
            raise ValueError(f"{field} cannot be empty.")

        overlap = used_ids & held_out_ids
        if overlap:
            raise ValueError(
                f"{field} overlaps current validation/test facilities: "
                f"{sorted(overlap)[:5]}"
            )

    config = checkpoint["config"]

    if config["embedding_dim"] != DEM_EMB_DIM:
        raise ValueError("Unexpected DEM embedding dimension.")

    if tuple(config["target_shape"]) != DEM_TARGET_SHAPE:
        raise ValueError("Unexpected DEM target shape.")

    scale = float(checkpoint["train_scale_m"])
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid checkpoint normalization scale.")

    terrain = TerrainCNN(
        embedding_dim=DEM_EMB_DIM,
        num_classes=N_CLASSES,
    )
    terrain.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    encoder = TerrainFeatureEncoder(
        terrain,
        freeze=freeze,
    ).to(device)

    encoder.config = {
        "architecture": "TerrainCNN",
        "initialization": "verified pretrained DEM checkpoint",
        "test_fold": test_fold,
        "freeze_encoder": freeze,
        "batchnorm": (
            "fixed running statistics"
            if freeze
            else "updated during training"
        ),
        "output_dim": DEM_EMB_DIM,
        "normalization": "checkpoint training scale",
        "train_scale_m": scale,
    }

    return encoder, scale

## 6. Dataset and DataLoaders
**Owner: Team — TBD**

Each sample returns image, tabular, DEM, label, and facility ID. Build train, validation, and test loaders with matching modality order.

In [16]:
# Stage DEM inputs from S3 once, then reuse them across folds/epochs.
import tempfile

# Keep the temporary directory alive for this notebook session.
# Rerunning this cell reuses previously staged files.
if "DEM_STAGING_ROOT" not in globals():
    DEM_STAGING_ROOT = tempfile.TemporaryDirectory(
        prefix="atlas-multimodal-dem-"
    )
    DEM_STAGED_SOURCES = {}

DEM_LOCAL_DIR = Path(DEM_STAGING_ROOT.name)


def stage_dem_files(manifest, dem_dir):
    dem_dir = Path(dem_dir)
    dem_dir.mkdir(parents=True, exist_ok=True)

    required = {"id", "dem_key"}
    if not required.issubset(manifest.columns):
        raise ValueError("Manifest requires id and dem_key columns.")

    if not manifest["id"].is_unique:
        raise ValueError("Manifest contains duplicate facility IDs.")

    downloaded = 0
    reused = 0

    for row in manifest[["id", "dem_key"]].itertuples(index=False):
        facility_id = str(row.id)

        if not (
            len(facility_id) == 10
            and facility_id.isascii()
            and facility_id.isdigit()
        ):
            raise ValueError(f"Invalid facility ID: {facility_id}")

        destination = dem_dir / f"{facility_id}_dem.tif"
        source = (S3_BUCKET, row.dem_key)
        cache_key = str(destination)

        if (
            DEM_STAGED_SOURCES.get(cache_key) == source
            and destination.is_file()
            and destination.stat().st_size > 0
        ):
            reused += 1
            continue

        partial = destination.with_suffix(".tif.part")

        try:
            s3_shared.download_file(
                S3_BUCKET,
                row.dem_key,
                str(partial),
            )

            if partial.stat().st_size == 0:
                raise ValueError("Downloaded DEM is empty.")

            # Expose the final filename only after download succeeds.
            partial.replace(destination)
            DEM_STAGED_SOURCES[cache_key] = source
            downloaded += 1

        except Exception as error:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                f"Could not stage DEM for {facility_id}: "
                f"s3://{S3_BUCKET}/{row.dem_key}"
            ) from error

    print("DEM staging directory:", dem_dir)
    print(f"Downloaded: {downloaded} | Reused: {reused}")
    return dem_dir


DEM_LOCAL_DIR = stage_dem_files(
    MULTIMODAL_MANIFEST,
    DEM_LOCAL_DIR,
)

DEM staging directory: C:\Users\kevin\AppData\Local\Temp\atlas-multimodal-dem-axxgvnv8
Downloaded: 180 | Reused: 0


In [18]:
# Create replacement DEM checkpoints compatible with the shared splits.
# Run once; this trains and uploads five new checkpoints.
from datetime import datetime, timezone

DEM_PRETRAIN_EPOCHS = 30

run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
DEM_CHECKPOINT_PREFIX = (
    "artifacts/usgs_terrain_baselines/"
    f"multimodal_compatible/{run_id}/cnn/"
)

for test_fold in range(1, N_FOLDS + 1):
    fold_seed = SEED + test_fold
    torch.manual_seed(fold_seed)

    split_df = TABULAR_FOLDS[test_fold]["split_df"]
    train_df = split_df.loc[
        split_df["split"] == "train"
    ].reset_index(drop=True)

    train_ids = train_df["id"].tolist()
    held_out_ids = set(
        split_df.loc[
            split_df["split"].isin(["val", "test"]), "id"
        ]
    )

    assert train_ids
    assert set(train_ids).isdisjoint(held_out_ids)

    # Fit normalization using exactly these training facilities.
    scale = fit_dem_scale(train_df, DEM_LOCAL_DIR)

    dem_dataset = USGSDemDataset(
        train_df,
        dem_dir=DEM_LOCAL_DIR,
        scale=scale,
    )

    # Cache prepared training DEMs once instead of reading TIFFs every epoch.
    samples = []
    for index in range(len(dem_dataset)):
        sample = dem_dataset[index]
        assert sample["id"] == train_ids[index]
        sample["label"] = torch.tensor(
            int(train_df.iloc[index]["size_class"]),
            dtype=torch.long,
        )
        samples.append(sample)

    train_loader = DataLoader(
        samples,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(fold_seed),
    )

    terrain = TerrainCNN(
        embedding_dim=DEM_EMB_DIM,
        num_classes=N_CLASSES,
    ).to(device)

    optimizer = torch.optim.AdamW(
        terrain.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()

    seen_ids = set()
    history = []

    for epoch in range(1, DEM_PRETRAIN_EPOCHS + 1):
        terrain.train()
        total_loss, total = 0.0, 0

        for batch in train_loader:
            seen_ids.update(batch["id"])

            optimizer.zero_grad(set_to_none=True)

            logits = terrain(
                batch["image"].to(device),
                batch["mask"].to(device),
            )
            labels = batch["label"].to(device)
            loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Nonfinite DEM loss in fold {test_fold}."
                )

            loss.backward()
            optimizer.step()

            count = labels.size(0)
            total_loss += loss.detach().item() * count
            total += count

        epoch_loss = total_loss / total
        history.append({
            "epoch": epoch,
            "train_loss": epoch_loss,
        })

        print(
            f"DEM fold {test_fold}/{N_FOLDS} | "
            f"epoch {epoch}/{DEM_PRETRAIN_EPOCHS} | "
            f"loss={epoch_loss:.4f}",
            flush=True,
        )

    assert seen_ids == set(train_ids)
    assert seen_ids.isdisjoint(held_out_ids)

    checkpoint = {
        "model_state_dict": {
            name: value.detach().cpu().clone()
            for name, value in terrain.state_dict().items()
        },
        "fold": test_fold,
        "train_scale_m": scale,
        "train_ids": sorted(seen_ids),
        "normalization_fit_ids": train_ids,
        "selection_ids": [],
        "config": {
            "embedding_dim": DEM_EMB_DIM,
            "target_shape": list(DEM_TARGET_SHAPE),
            "num_classes": N_CLASSES,
            "seed": fold_seed,
            "epochs": DEM_PRETRAIN_EPOCHS,
            "initialization": "random",
            "selection_method": "fixed final epoch",
            "loss": "unweighted cross entropy",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
        },
        "history": history,
        "spatial_folds_key": SPATIAL_FOLDS_KEY,
    }

    buffer = io.BytesIO()
    torch.save(checkpoint, buffer)
    buffer.seek(0)

    checkpoint_key = (
        f"{DEM_CHECKPOINT_PREFIX}"
        f"terrain_cnn_fold_{test_fold}.pt"
    )
    s3_shared.upload_fileobj(
        buffer,
        S3_BUCKET,
        checkpoint_key,
        ExtraArgs={"ContentType": "application/octet-stream"},
    )

    print(f"Saved: s3://{S3_BUCKET}/{checkpoint_key}")
    del terrain, optimizer, train_loader, samples, dem_dataset

print("Replacement checkpoint prefix:", DEM_CHECKPOINT_PREFIX)

DEM fold 1/5 | epoch 1/30 | loss=1.0591
DEM fold 1/5 | epoch 2/30 | loss=0.9097
DEM fold 1/5 | epoch 3/30 | loss=0.8082
DEM fold 1/5 | epoch 4/30 | loss=0.7471
DEM fold 1/5 | epoch 5/30 | loss=0.7281
DEM fold 1/5 | epoch 6/30 | loss=0.7238
DEM fold 1/5 | epoch 7/30 | loss=0.7263
DEM fold 1/5 | epoch 8/30 | loss=0.7155
DEM fold 1/5 | epoch 9/30 | loss=0.6995
DEM fold 1/5 | epoch 10/30 | loss=0.6922
DEM fold 1/5 | epoch 11/30 | loss=0.6855
DEM fold 1/5 | epoch 12/30 | loss=0.6849
DEM fold 1/5 | epoch 13/30 | loss=0.6811
DEM fold 1/5 | epoch 14/30 | loss=0.6618
DEM fold 1/5 | epoch 15/30 | loss=0.6787
DEM fold 1/5 | epoch 16/30 | loss=0.6520
DEM fold 1/5 | epoch 17/30 | loss=0.6659
DEM fold 1/5 | epoch 18/30 | loss=0.6306
DEM fold 1/5 | epoch 19/30 | loss=0.6373
DEM fold 1/5 | epoch 20/30 | loss=0.6415
DEM fold 1/5 | epoch 21/30 | loss=0.6039
DEM fold 1/5 | epoch 22/30 | loss=0.6250
DEM fold 1/5 | epoch 23/30 | loss=0.6063
DEM fold 1/5 | epoch 24/30 | loss=0.6200
DEM fold 1/5 | epoch 25/3

In [19]:
#reads checkpoints from s3 into RAM and checks it against the matching multimodal folds
#also checks IDs used for normalization and checkpoint selection to prevent leakage

DEM_CHECKPOINT_AUDITS = []
VERIFIED_DEM_CHECKPOINTS = {}

PROVENANCE_FIELDS = (
    "train_ids",
    "normalization_fit_ids",
    "selection_ids",
)


def checkpoint_id_set(values):
    if not isinstance(values, (list, tuple)):
        raise ValueError("Provenance IDs must be a list or tuple.")

    if not values:
        return set()

    normalized = normalize_facility_ids(
        pd.Series(values, dtype="string")
    )
    return set(normalized.tolist())


for test_fold in range(1, N_FOLDS + 1):
    split_df = TABULAR_FOLDS[test_fold]["split_df"]
    held_out_ids = set(
        split_df.loc[
            split_df["split"].isin(["val", "test"]),
            "id",
        ]
    )

    checkpoint_key = (
        f"{DEM_CHECKPOINT_PREFIX.rstrip('/')}/"
        f"terrain_cnn_fold_{test_fold}.pt"
    )

    record = {
        "test_fold": test_fold,
        "checkpoint_key": checkpoint_key,
        "status": "UNVERIFIED",
        "missing_fields": "",
        "overlapping_facilities": 0,
        "reason": "",
    }

    try:
        body = s3_shared.get_object(
            Bucket=S3_BUCKET,
            Key=checkpoint_key,
        )["Body"]

        try:
            checkpoint = torch.load(
                io.BytesIO(body.read()),
                map_location="cpu",
                weights_only=True,
            )
        finally:
            body.close()

        if int(checkpoint["fold"]) != test_fold:
            raise ValueError("Checkpoint fold number does not match.")

        config = checkpoint["config"]
        if config["embedding_dim"] != DEM_EMB_DIM:
            raise ValueError("Unexpected DEM embedding dimension.")

        if tuple(config["target_shape"]) != DEM_TARGET_SHAPE:
            raise ValueError("Unexpected DEM target shape.")

        scale = float(checkpoint["train_scale_m"])
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError("Invalid DEM normalization scale.")

        missing = [
            field
            for field in PROVENANCE_FIELDS
            if field not in checkpoint
        ]
        record["missing_fields"] = ", ".join(missing)

        overlaps = set()

        for field in PROVENANCE_FIELDS:
            if field not in checkpoint:
                continue

            used_ids = checkpoint_id_set(checkpoint[field])

            if field in ("train_ids", "normalization_fit_ids"):
                if not used_ids:
                    raise ValueError(f"{field} cannot be empty.")

            overlaps.update(used_ids & held_out_ids)

        record["overlapping_facilities"] = len(overlaps)

        if overlaps:
            record["status"] = "REJECTED"
            record["reason"] = (
                "Checkpoint preparation used current validation/test "
                f"facilities. Examples: {sorted(overlaps)[:5]}"
            )
        elif missing:
            record["reason"] = (
                "Missing provenance; absence of leakage cannot be verified."
            )
        else:
            record["status"] = "PASS"
            record["reason"] = (
                "Recorded training, normalization, and selection IDs "
                "exclude current validation/test facilities."
            )
            VERIFIED_DEM_CHECKPOINTS[test_fold] = checkpoint

    except Exception as error:
        record["status"] = "ERROR"
        record["reason"] = str(error)

    DEM_CHECKPOINT_AUDITS.append(record)

dem_checkpoint_report = pd.DataFrame(DEM_CHECKPOINT_AUDITS)
display(dem_checkpoint_report)

print(
    "Verified DEM checkpoints:",
    f"{len(VERIFIED_DEM_CHECKPOINTS)}/{N_FOLDS}",
)

,test_fold,checkpoint_key,status,missing_fields,overlapping_facilities,reason
0,1,artifacts/usgs_terrain_baselines/multimodal_co...,PASS,,0,"Recorded training, normalization, and selectio..."
1,2,artifacts/usgs_terrain_baselines/multimodal_co...,PASS,,0,"Recorded training, normalization, and selectio..."
2,3,artifacts/usgs_terrain_baselines/multimodal_co...,PASS,,0,"Recorded training, normalization, and selectio..."
3,4,artifacts/usgs_terrain_baselines/multimodal_co...,PASS,,0,"Recorded training, normalization, and selectio..."
4,5,artifacts/usgs_terrain_baselines/multimodal_co...,PASS,,0,"Recorded training, normalization, and selectio..."


Verified DEM checkpoints: 5/5


In [20]:

# Section 6 — Shared multimodal Dataset and DataLoaders

class MultimodalDataset(Dataset):
    """Facility-ID-aligned NAIP, tabular, DEM, and labels."""

    def __init__(self, dataframe, dem_dir, dem_scale,
                 tabular_lookup=None):
        self.df = dataframe.reset_index(drop=True).copy()

        assert self.df["id"].is_unique
        assert self.df["size_class"].isin(range(N_CLASSES)).all()

        self.dem_dataset = USGSDemDataset(
            self.df,
            dem_dir=dem_dir,
            scale=dem_scale,
        )

        # Explicit dependency on Swati's fitted preprocessing.
        self.tabular_lookup = tabular_lookup

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        facility_id = str(row["id"])

        image = load_image_tensor(row["image_key"])
        dem_sample = self.dem_dataset[index]

        assert dem_sample["id"] == facility_id

        sample = {
            "id": facility_id,
            "image": image,
            "dem": dem_sample["image"],
            "dem_mask": dem_sample["mask"],
            "label": torch.tensor(
                int(row["size_class"]),
                dtype=torch.long,
            ),
        }

        if self.tabular_lookup is not None:
            tabular = self.tabular_lookup([facility_id])
            assert tabular.ndim == 2 and tabular.shape[0] == 1
            sample["tabular"] = tabular[0].float()

        return sample


def build_multimodal_loaders(
    split_df,
    dem_dir,
    tabular_context,
    dem_scale,
    batch_size=BATCH_SIZE,
    num_workers=0,
):
    """Build geographically separated loaders for one outer fold."""

    if tabular_context is None:
        raise ValueError("Provide this fold's tabular preprocessing.")
    
    #reject preprocessing prepared for different split assignments.
    for split in ("train", "val", "test"):
        expected_ids = set(
            split_df.loc[split_df["split"] == split, "id"]
        )
        prepared_ids = set(
            tabular_context["ids_by_split"][split]
        )

        if expected_ids != prepared_ids:
            raise ValueError(
                f"Tabular preprocessing does not match the {split} split."
            )

    expected_train_ids = set(
        split_df.loc[split_df["split"] == "train", "id"]
    )
    if set(tabular_context["fit_ids"]) != expected_train_ids:
        raise ValueError(
            "Tabular preprocessing was fitted on different training IDs."
        )

    tabular_lookup = tabular_context["lookup"]
            

    dem_dir = Path(dem_dir)
    assert dem_dir.is_dir(), f"Missing DEM directory: {dem_dir}"

    # Preserve the normalization expected by the pretrained DEM encoder.
    dem_scale = float(dem_scale)

    if not np.isfinite(dem_scale) or dem_scale <= 0:
        raise ValueError(
            "Provide a positive, finite scale returned by "
            "load_terrain_encoder()."
        )

    loaders = {}

    for split in ("train", "val", "test"):
        subset = split_df.loc[
            split_df["split"] == split
        ].copy()

        dataset = MultimodalDataset(
            dataframe=subset,
            dem_dir=dem_dir,
            dem_scale=dem_scale,
            tabular_lookup=tabular_lookup,
        )

        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
        )

    return loaders, dem_scale


print("MultimodalDataset defined.")
print("DataLoader factory defined.")
print("Local DEM directory:", DEM_LOCAL_DIR)


MultimodalDataset defined.
DataLoader factory defined.
Local DEM directory: C:\Users\kevin\AppData\Local\Temp\atlas-multimodal-dem-axxgvnv8


## 7. Feature concatenation and classifier
**Owner: Team — TBD**

Concatenate along `dim=1`, in image/tabular/DEM order:

**`[B, 512] + [B, 64] + [B, 64] → [B, 640] → classifier → [B, K]`**

`K` = number of classes. Return logits for cross-entropy loss.

In [26]:
MODALITY_SETS = {
    "image_only": ("image",),
    "image_tabular": ("image", "tabular"),
    "image_dem": ("image", "dem"),
    "all": ("image", "tabular", "dem"),
}


class MultimodalFusion(nn.Module):
    def __init__(
        self,
        image_encoder,
        tabular_encoder=None,
        dem_encoder=None,
        hidden_dim=128,
        dropout=0.3,
        variant="all",
    ):
        super().__init__()

        if variant not in MODALITY_SETS:
            raise ValueError(f"Unknown variant: {variant}")

        self.modalities = MODALITY_SETS[variant]
        self.image_encoder = image_encoder
        self.tabular_encoder = (
            tabular_encoder if "tabular" in self.modalities else None
        )
        self.dem_encoder = (
            dem_encoder if "dem" in self.modalities else None
        )

        if "tabular" in self.modalities and tabular_encoder is None:
            raise ValueError("This variant requires tabular features.")
        if "dem" in self.modalities and dem_encoder is None:
            raise ValueError("This variant requires DEM features.")

        self.fusion_dim = (
            IMAGE_EMB_DIM
            + (TAB_EMB_DIM if "tabular" in self.modalities else 0)
            + (DEM_EMB_DIM if "dem" in self.modalities else 0)
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(self.fusion_dim),
            nn.Linear(self.fusion_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, N_CLASSES),
        )

    def forward(self, images, tabular, dem, dem_mask):
        image_features = self.image_encoder(images)
        assert image_features.shape == (
            images.shape[0], IMAGE_EMB_DIM
        )
        features = [image_features]

        if "tabular" in self.modalities:
            tabular_features = self.tabular_encoder(tabular)
            assert tabular_features.shape == (
                images.shape[0], TAB_EMB_DIM
            )
            features.append(tabular_features)

        if "dem" in self.modalities:
            dem_features = self.dem_encoder.encode(dem, dem_mask)
            assert dem_features.shape == (
                images.shape[0], DEM_EMB_DIM
            )
            features.append(dem_features)

        return self.classifier(torch.cat(features, dim=1))


print("Fusion variants:", MODALITY_SETS)

Fusion variants: {'image_only': ('image',), 'image_tabular': ('image', 'tabular'), 'image_dem': ('image', 'dem'), 'all': ('image', 'tabular', 'dem')}


## 8. Integration check
**Owner: Team — TBD**

Run one batch. Verify embedding/logit shapes, compute loss, and confirm gradients reach all intended trainable components.

In [27]:
# Section 8 — Real multimodal integration smoke test

smoke_fold = TEST_FOLD
fold_data = TABULAR_FOLDS[smoke_fold]

# Instantiate each encoder using this fold's configuration.
smoke_image_encoder = ImageEncoder(
    checkpoint_key=IMAGE_CHECKPOINT_KEY,
    freeze_backbone=IMAGE_FREEZE_BACKBONE,
)

smoke_tabular_encoder = TabularEncoder(
    in_dim=fold_data["tabular"]["input_dim"],
)

smoke_dem_encoder, smoke_scale = load_terrain_encoder(
    test_fold=smoke_fold,
    device=device,
    freeze=DEM_FREEZE_ENCODER,
)

smoke_model = MultimodalFusion(
    image_encoder=smoke_image_encoder,
    tabular_encoder=smoke_tabular_encoder,
    dem_encoder=smoke_dem_encoder,
).to(device)

smoke_loaders, _ = build_multimodal_loaders(
    split_df=fold_data["split_df"],
    dem_dir=DEM_LOCAL_DIR,
    tabular_context=fold_data["tabular"],
    dem_scale=smoke_scale,
)

batch = next(iter(smoke_loaders["train"]))
batch_size = len(batch["id"])

# Verify facility membership, labels, and tabular alignment.
training_rows = fold_data["split_df"].loc[
    fold_data["split_df"]["split"] == "train"
].set_index("id")

assert batch_size > 0
assert len(set(batch["id"])) == batch_size
assert set(batch["id"]).issubset(training_rows.index)

expected_labels = torch.tensor(
    training_rows.loc[batch["id"], "size_class"].to_numpy(),
    dtype=torch.long,
)
assert torch.equal(batch["label"], expected_labels)
assert torch.equal(
    batch["tabular"],
    fold_data["tabular"]["lookup"](batch["id"]),
)

expected_inputs = {
    "image": (batch_size, 3, 224, 224),
    "tabular": (
        batch_size,
        fold_data["tabular"]["input_dim"],
    ),
    "dem": (batch_size, 1, *DEM_TARGET_SHAPE),
    "dem_mask": (batch_size, 1, *DEM_TARGET_SHAPE),
}

for name, shape in expected_inputs.items():
    assert tuple(batch[name].shape) == shape, name
    assert torch.isfinite(batch[name]).all(), name

assert (batch["dem_mask"].flatten(1).sum(1) > 0).all()
assert batch["label"].dtype == torch.long

inputs = {
    name: batch[name].to(device)
    for name in expected_inputs
}
labels = batch["label"].to(device)

# Observe the actual forward pass without running encoders twice.
observed_shapes = {}


def capture_shape(name):
    def hook(module, arguments, output):
        observed_shapes[name] = tuple(output.shape)
    return hook


hooks = [
    smoke_model.image_encoder.register_forward_hook(
        capture_shape("image")
    ),
    smoke_model.tabular_encoder.register_forward_hook(
        capture_shape("tabular")
    ),
    smoke_model.dem_encoder.terrain.embedding.register_forward_hook(
        capture_shape("dem")
    ),
    smoke_model.classifier[0].register_forward_hook(
        capture_shape("fused")
    ),
]

branches = {
    "image": (
        smoke_model.image_encoder,
        IMAGE_FREEZE_BACKBONE,
    ),
    "tabular": (
        smoke_model.tabular_encoder,
        False,
    ),
    "dem": (
        smoke_model.dem_encoder,
        DEM_FREEZE_ENCODER,
    ),
    "classifier": (
        smoke_model.classifier,
        False,
    ),
}

# Record buffers of frozen encoders before entering training mode.
frozen_buffers = {
    name: {
        key: value.detach().clone()
        for key, value in branch.named_buffers()
    }
    for name, (branch, frozen) in branches.items()
    if frozen
}

try:
    smoke_model.train()
    smoke_model.zero_grad(set_to_none=True)

    logits = smoke_model(
        inputs["image"],
        inputs["tabular"],
        inputs["dem"],
        inputs["dem_mask"],
    )

    assert observed_shapes == {
        "image": (batch_size, IMAGE_EMB_DIM),
        "tabular": (batch_size, TAB_EMB_DIM),
        "dem": (batch_size, DEM_EMB_DIM),
        "fused": (batch_size, FUSION_DIM),
    }, observed_shapes

    assert tuple(logits.shape) == (batch_size, N_CLASSES)
    assert torch.isfinite(logits).all()

    loss = nn.CrossEntropyLoss()(logits, labels)
    assert torch.isfinite(loss), "Nonfinite loss"

    loss.backward()

    gradient_status = {}

    for name, (branch, frozen) in branches.items():
        parameters = list(branch.parameters())

        if frozen:
            assert all(not p.requires_grad for p in parameters), name
            assert all(p.grad is None for p in parameters), name

            assert all(
                not module.training
                for module in branch.modules()
                if isinstance(
                    module,
                    nn.modules.batchnorm._BatchNorm,
                )
            ), f"{name}: frozen BatchNorm entered training mode"

            for key, value in branch.named_buffers():
                assert torch.equal(
                    frozen_buffers[name][key], value
                ), f"{name}: frozen buffer changed: {key}"

            gradient_status[name] = "frozen; no gradients"
        else:
            trainable = [
                p for p in parameters if p.requires_grad
            ]
            assert trainable, f"{name}: no trainable parameters"
            assert all(
                p.grad is not None
                and torch.isfinite(p.grad).all()
                for p in trainable
            ), f"{name}: missing or nonfinite gradients"

            assert any(
                torch.count_nonzero(p.grad).item() > 0
                for p in trainable
            ), f"{name}: all gradients are zero"

            gradient_status[name] = "finite gradients received"

    SMOKE_TEST_REPORT = {
        "fold": smoke_fold,
        "batch_size": batch_size,
        "embedding_shapes": observed_shapes,
        "logits_shape": tuple(logits.shape),
        "loss": loss.detach().item(),
        "gradients": gradient_status,
        "status": "PASS",
    }

    print("Embedding shapes:", observed_shapes)
    print("Logits:", tuple(logits.shape))
    print(f"Loss: {loss.detach().item():.4f}")

    for name, status in gradient_status.items():
        print(f"{name}: {status}")

    print("Multimodal integration smoke test: PASS")

finally:
    for handle in hooks:
        handle.remove()

    smoke_model.zero_grad(set_to_none=True)

Embedding shapes: {'image': (16, 512), 'tabular': (16, 64), 'dem': (16, 64), 'fused': (16, 640)}
Logits: (16, 3)
Loss: 1.1772
image: frozen; no gradients
tabular: finite gradients received
dem: frozen; no gradients
classifier: finite gradients received
Multimodal integration smoke test: PASS


## 9. Training
**Owner: Team — TBD**

Define loss, optimizer, scheduler, and epochs. Track train/validation loss and macro-F1; use validation for early stopping and best-checkpoint selection. Save weights and model configuration.

In [28]:
#epoch helper

import json
import pickle
import sklearn
from datetime import datetime, timezone
from sklearn.metrics import accuracy_score, f1_score


def run_multimodal_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss, total = 0.0, 0
    actual, predicted = [], []

    with torch.set_grad_enabled(training):
        for batch in loader:
            if training:
                optimizer.zero_grad(set_to_none=True)

            labels = batch["label"].to(device)

            logits = model(
                batch["image"].to(device),
                batch["tabular"].to(device),
                batch["dem"].to(device),
                batch["dem_mask"].to(device),
            )
            loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite multimodal loss.")

            if training:
                loss.backward()
                optimizer.step()

            count = labels.size(0)
            total_loss += loss.detach().item() * count
            total += count

            actual.extend(batch["label"].tolist())
            predicted.extend(
                logits.detach().argmax(1).cpu().tolist()
            )

    if total == 0:
        raise ValueError("Cannot evaluate an empty loader.")

    return {
        "loss": total_loss / total,
        "accuracy": float(accuracy_score(actual, predicted)),
        # Fixed three-class macro-F1; undefined F1 values count as zero.
        "macro_f1": float(f1_score(
            actual,
            predicted,
            labels=list(range(N_CLASSES)),
            average="macro",
            zero_division=0,
        )),
    }

In [29]:
#train multimodal (one fold) and save its best checkpoint
#cached into CPU RAM avoiding repeated s3 downloads every epoch

def train_multimodal_fold(test_fold, run_prefix,variant="all"):
    if MAX_EPOCHS < 1 or EARLY_STOPPING_PATIENCE < 1:
        raise ValueError("Epochs and patience must be positive.")

    fold_seed = SEED + test_fold
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    fold_data = TABULAR_FOLDS[test_fold]
    context = fold_data["tabular"]

    # Fresh model components for every fold.
    dem_encoder, scale = load_terrain_encoder(
        test_fold=test_fold,
        device=device,
        freeze=DEM_FREEZE_ENCODER,
    )

    model = MultimodalFusion(
        image_encoder=ImageEncoder(
            checkpoint_key=IMAGE_CHECKPOINT_KEY,
            freeze_backbone=IMAGE_FREEZE_BACKBONE,
        ),
        tabular_encoder=TabularEncoder(
            in_dim=context["input_dim"],
            emb_dim=TAB_EMB_DIM,
            hidden=128,
            dropout=0.1,
        ),
        dem_encoder=dem_encoder,
        hidden_dim=128,
        dropout=0.3,
        variant=variant,
    ).to(device)

    source_loaders, _ = build_multimodal_loaders(
        split_df=fold_data["split_df"],
        dem_dir=DEM_LOCAL_DIR,
        tabular_context=context,
        dem_scale=scale,
    )

    # Cache training/validation inputs only. Do not read test samples.
    cached_loaders = {}
    for split in ("train", "val"):
        dataset = source_loaders[split].dataset
        samples = [dataset[index] for index in range(len(dataset))]

        cached_loaders[split] = DataLoader(
            samples,
            batch_size=BATCH_SIZE,
            shuffle=(split == "train"),
            num_workers=0,
            generator=torch.Generator().manual_seed(fold_seed),
        )

    # Match the unweighted cross-entropy used in the integration test.
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    history = []
    best_loss = float("inf")
    best_weights = None
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_multimodal_epoch(
            model, cached_loaders["train"], criterion, optimizer
        )
        val_metrics = run_multimodal_epoch(
            model, cached_loaders["val"], criterion
        )

        history.append({
            "epoch": epoch,
            "learning_rate": optimizer.param_groups[0]["lr"],
            **{
                f"train_{name}": value
                for name, value in train_metrics.items()
            },
            **{
                f"val_{name}": value
                for name, value in val_metrics.items()
            },
        })

        print(
            f"Fold {test_fold}/{N_FOLDS} | "
            f"epoch {epoch}/{MAX_EPOCHS} | "
            f"train loss={train_metrics['loss']:.4f} | "
            f"val loss={val_metrics['loss']:.4f} | "
            f"val macro-F1={val_metrics['macro_f1']:.4f}",
            flush=True,
        )

        if val_metrics["loss"] < best_loss:
            best_loss = val_metrics["loss"]
            best_epoch = epoch
            stale_epochs = 0
            best_weights = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
        else:
            stale_epochs += 1

        scheduler.step(val_metrics["loss"])

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    if best_weights is None:
        raise RuntimeError("No valid multimodal checkpoint produced.")

    model.load_state_dict(best_weights)
    model.eval()

    dem_audit = dem_checkpoint_report.loc[
        dem_checkpoint_report["test_fold"] == test_fold
    ].iloc[0]

    config = {
        "test_fold": test_fold,
        "seed": fold_seed,
        "class_names": list(CLASS_NAMES),
        "tabular_input_dim": context["input_dim"],
        "tabular_embedding_dim": TAB_EMB_DIM,
        "tabular_hidden": 128,
        "tabular_dropout": 0.1,
        "fusion_hidden": 128,
        "fusion_dropout": 0.3,
        "image_encoder": dict(IMAGE_ENCODER_CONFIG),
        "dem_encoder": {
            **dem_encoder.config,
            "checkpoint_key": str(dem_audit["checkpoint_key"]),
        },
        "tabular_feature_names": context["feature_names"],
        "tabular_input_columns": list(
            context["preprocessor"].feature_names_in_
        ),
        "tabular_dropped_constant": context["dropped_constant"],
        "tabular_fit_ids": list(context["fit_ids"]),
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "loss": "unweighted CrossEntropyLoss",
        "scheduler": {
            "name": "ReduceLROnPlateau",
            "monitor": "val_loss",
            "factor": 0.5,
            "patience": 2,
        },
        "checkpoint_monitor": "val_loss",
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_val_loss": best_loss,
        "macro_f1_policy": "fixed classes 0,1,2; zero_division=0",
        "spatial_folds_key": SPATIAL_FOLDS_KEY,
        "tabular_source_key": TABULAR_KEY,
        "torch_version": str(torch.__version__),
        "sklearn_version": sklearn.__version__,
        "variant": variant,
        "modalities": list(MODALITY_SETS[variant]),
        "fusion_dim": model.fusion_dim,
    }

    fold_prefix = (
    f"{run_prefix}fold_{test_fold}/"
    if variant == "all"
    else f"{run_prefix}ablations/{variant}/fold_{test_fold}/"
    )

    def upload(name, payload, content_type):
        s3_shared.upload_fileobj(
            io.BytesIO(payload),
            S3_BUCKET,
            fold_prefix + name,
            ExtraArgs={"ContentType": content_type},
        )

    buffer = io.BytesIO()
    torch.save({
        "model_state_dict": best_weights,
        "config": config,
    }, buffer)

    upload("model.pkl", buffer.getvalue(),
           "application/octet-stream")
    upload("config.json", json.dumps(config, indent=2).encode(),
           "application/json")
    upload("training_history.csv",
           pd.DataFrame(history).to_csv(index=False).encode(),
           "text/csv")
    upload("data_splits.csv",
           fold_data["split_df"].to_csv(index=False).encode(),
           "text/csv")
    upload(
        "tabular_preprocessor.pkl",
        pickle.dumps(
            context["preprocessor"],
            protocol=pickle.HIGHEST_PROTOCOL,
        ),
        "application/octet-stream",
    )

    print(
        f"Restored epoch {best_epoch}; "
        f"saved to s3://{S3_BUCKET}/{fold_prefix}"
    )

    return {
        "test_fold": test_fold,
        "best_epoch": best_epoch,
        "epochs_completed": len(history),
        "best_val_loss": best_loss,
        "checkpoint_key": fold_prefix + "model.pkl",
        "artifact_prefix": fold_prefix,
    }

In [30]:
#run all 5 folds
if SMOKE_TEST_REPORT["status"] != "PASS":
    raise RuntimeError("Complete Section 8 successfully first.")

if set(VERIFIED_DEM_CHECKPOINTS) != set(
    range(1, N_FOLDS + 1)
):
    raise RuntimeError("Verify every DEM checkpoint before training.")

MULTIMODAL_RUN_NAME = datetime.now(timezone.utc).strftime(
    "multimodal-%Y%m%d-%H%M%S-%f"
)
MULTIMODAL_RUN_PREFIX = (
    f"artifacts/multimodal/{MULTIMODAL_RUN_NAME}/"
)

FOLD_TRAINING_RESULTS = {}

for test_fold in range(1, N_FOLDS + 1):
    FOLD_TRAINING_RESULTS[test_fold] = train_multimodal_fold(
        test_fold,
        MULTIMODAL_RUN_PREFIX,
    )

display(pd.DataFrame(FOLD_TRAINING_RESULTS.values()))

print(
    "Training artifacts:",
    f"s3://{S3_BUCKET}/{MULTIMODAL_RUN_PREFIX}",
)

Fold 1/5 | epoch 1/30 | train loss=0.8361 | val loss=0.8349 | val macro-F1=0.2688
Fold 1/5 | epoch 2/30 | train loss=0.5256 | val loss=0.7377 | val macro-F1=0.3579
Fold 1/5 | epoch 3/30 | train loss=0.4156 | val loss=0.8256 | val macro-F1=0.2688
Fold 1/5 | epoch 4/30 | train loss=0.3527 | val loss=0.8392 | val macro-F1=0.2688
Fold 1/5 | epoch 5/30 | train loss=0.2382 | val loss=0.8582 | val macro-F1=0.4302
Fold 1/5 | epoch 6/30 | train loss=0.1895 | val loss=0.9684 | val macro-F1=0.3333
Fold 1/5 | epoch 7/30 | train loss=0.1539 | val loss=0.9779 | val macro-F1=0.3850
Fold 1/5 | epoch 8/30 | train loss=0.1195 | val loss=0.9807 | val macro-F1=0.3850
Fold 1/5 | epoch 9/30 | train loss=0.1071 | val loss=1.0064 | val macro-F1=0.3850
Early stopping at epoch 9.
Restored epoch 2; saved to s3://atlas-acm-research-f26/artifacts/multimodal/multimodal-20261104-062610-466211/fold_1/
Fold 2/5 | epoch 1/30 | train loss=0.8135 | val loss=0.6153 | val macro-F1=0.2732
Fold 2/5 | epoch 2/30 | train loss=

,test_fold,best_epoch,epochs_completed,best_val_loss,checkpoint_key,artifact_prefix
0,1,2,9,0.737727,artifacts/multimodal/multimodal-20261104-06261...,artifacts/multimodal/multimodal-20261104-06261...
1,2,3,10,0.556634,artifacts/multimodal/multimodal-20261104-06261...,artifacts/multimodal/multimodal-20261104-06261...
2,3,5,12,0.628294,artifacts/multimodal/multimodal-20261104-06261...,artifacts/multimodal/multimodal-20261104-06261...
3,4,4,11,0.749404,artifacts/multimodal/multimodal-20261104-06261...,artifacts/multimodal/multimodal-20261104-06261...
4,5,2,9,0.568547,artifacts/multimodal/multimodal-20261104-06261...,artifacts/multimodal/multimodal-20261104-06261...


Training artifacts: s3://atlas-acm-research-f26/artifacts/multimodal/multimodal-20261104-062610-466211/


## 10. Evaluation
**Owner: Team — TBD**

Evaluate the best checkpoint on held-out test facilities. Report accuracy, macro-F1, per-class precision/recall/F1, and a confusion matrix. Export predictions.

Compare on the same test facilities: image only, image + tabular, image + DEM, and all three modalities.

In [31]:
#artifact and model-loading helpers

import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


def read_artifact(key):
    body = s3_shared.get_object(
        Bucket=S3_BUCKET, Key=key
    )["Body"]
    try:
        return body.read()
    finally:
        body.close()


def check_saved_split(prefix, test_fold):
    splits = pd.read_csv(
        io.BytesIO(read_artifact(prefix + "data_splits.csv")),
        dtype={"id": str},
    )

    columns = ["id", "split", "size_class"]
    current = TABULAR_FOLDS[test_fold]["split_df"]

    pd.testing.assert_frame_equal(
        splits[columns].sort_values("id").reset_index(drop=True),
        current[columns].sort_values("id").reset_index(drop=True),
        check_dtype=False,
    )
    return splits


def load_comparison_model(variant, test_fold):
    saved = MODEL_RESULTS[variant][test_fold]
    prefix = saved["artifact_prefix"]

    checkpoint = torch.load(
        io.BytesIO(read_artifact(saved["checkpoint_key"])),
        map_location="cpu",
        weights_only=True,
    )
    config = checkpoint["config"]

    # Older full-model checkpoints did not contain a variant field.
    assert config.get("variant", "all") == variant
    assert config["test_fold"] == test_fold
    assert config["class_names"] == list(CLASS_NAMES)
    assert config["sklearn_version"] == sklearn.__version__
    assert config["image_encoder"]["initialization"] == IMAGE_WEIGHTS.name

    splits = check_saved_split(prefix, test_fold)

    # Restore training-time preprocessing; do not fit on test data.
    preprocessor = pickle.loads(
        read_artifact(prefix + "tabular_preprocessor.pkl")
    )
    aligned = tab_df.set_index("id").loc[splits["id"]]
    features = aligned[config["tabular_input_columns"]].copy()

    for name, transformer, columns in preprocessor.transformers_:
        if name == "cat":
            for column in columns:
                values = features[column].astype(object)
                features[column] = values.where(values.notna(), np.nan)

    matrix = preprocessor.transform(features).astype(np.float32)
    assert matrix.shape[1] == config["tabular_input_dim"]
    assert np.isfinite(matrix).all()

    row_of = {
        facility_id: index
        for index, facility_id in enumerate(splits["id"])
    }

    def lookup(ids):
        return torch.from_numpy(matrix[[row_of[i] for i in ids]])

    audit = dem_checkpoint_report.loc[
        dem_checkpoint_report["test_fold"] == test_fold
    ].iloc[0]
    assert audit["status"] == "PASS"
    assert (
        audit["checkpoint_key"]
        == config["dem_encoder"]["checkpoint_key"]
    )

    dem_encoder, scale = load_terrain_encoder(
        test_fold=test_fold,
        device=device,
        freeze=config["dem_encoder"]["freeze_encoder"],
    )
    assert np.isclose(scale, config["dem_encoder"]["train_scale_m"])

    model = MultimodalFusion(
        image_encoder=ImageEncoder(
            checkpoint_key=config["image_encoder"]["atlas_checkpoint_key"],
            freeze_backbone=config["image_encoder"]["freeze_backbone"],
        ),
        tabular_encoder=TabularEncoder(
            in_dim=config["tabular_input_dim"],
            emb_dim=config["tabular_embedding_dim"],
            hidden=config["tabular_hidden"],
            dropout=config["tabular_dropout"],
        ),
        dem_encoder=dem_encoder,
        hidden_dim=config["fusion_hidden"],
        dropout=config["fusion_dropout"],
        variant=variant,
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

    test_rows = splits.loc[splits["split"] == "test"].copy()
    assert set(test_rows["id"]).isdisjoint(config["tabular_fit_ids"])

    dataset = MultimodalDataset(
        test_rows,
        dem_dir=DEM_LOCAL_DIR,
        dem_scale=scale,
        tabular_lookup=lookup,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    return model, loader, config, prefix

In [32]:
#train the three other variants 
assert set(FOLD_TRAINING_RESULTS) == set(range(1, N_FOLDS + 1))

MODEL_RESULTS = {"all": dict(FOLD_TRAINING_RESULTS)}

settings = {
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
}

# Verify compatibility before starting any additional training.
for test_fold in range(1, N_FOLDS + 1):
    prefix = FOLD_TRAINING_RESULTS[test_fold]["artifact_prefix"]
    original = json.loads(read_artifact(prefix + "config.json"))

    check_saved_split(prefix, test_fold)

    for name, expected in settings.items():
        assert original[name] == expected, (
            f"Training setting changed: {name}"
        )

    assert original["image_encoder"] == IMAGE_ENCODER_CONFIG
    assert (
        original["dem_encoder"]["freeze_encoder"]
        == DEM_FREEZE_ENCODER
    )
    assert (
        original["tabular_feature_names"]
        == TABULAR_FOLDS[test_fold]["tabular"]["feature_names"]
    )

for variant in ("image_only", "image_tabular", "image_dem"):
    MODEL_RESULTS[variant] = {}

    for test_fold in range(1, N_FOLDS + 1):
        print(f"\n===== {variant}: fold {test_fold} =====")

        MODEL_RESULTS[variant][test_fold] = train_multimodal_fold(
            test_fold,
            MULTIMODAL_RUN_PREFIX,
            variant=variant,
        )

print("All four configurations have five trained checkpoints.")


===== image_only: fold 1 =====
Fold 1/5 | epoch 1/30 | train loss=0.8514 | val loss=0.7461 | val macro-F1=0.2688
Fold 1/5 | epoch 2/30 | train loss=0.5447 | val loss=0.6980 | val macro-F1=0.3579
Fold 1/5 | epoch 3/30 | train loss=0.4583 | val loss=0.7393 | val macro-F1=0.3711
Fold 1/5 | epoch 4/30 | train loss=0.4283 | val loss=0.7778 | val macro-F1=0.3711
Fold 1/5 | epoch 5/30 | train loss=0.3131 | val loss=0.7644 | val macro-F1=0.3452
Fold 1/5 | epoch 6/30 | train loss=0.2513 | val loss=0.8653 | val macro-F1=0.3711
Fold 1/5 | epoch 7/30 | train loss=0.2229 | val loss=0.8373 | val macro-F1=0.3579
Fold 1/5 | epoch 8/30 | train loss=0.2126 | val loss=0.8195 | val macro-F1=0.3988
Fold 1/5 | epoch 9/30 | train loss=0.1900 | val loss=0.8612 | val macro-F1=0.3579
Early stopping at epoch 9.
Restored epoch 2; saved to s3://atlas-acm-research-f26/artifacts/multimodal/multimodal-20261104-062610-466211/ablations/image_only/fold_1/

===== image_only: fold 2 =====
Fold 2/5 | epoch 1/30 | train lo

In [33]:
#evaluate all four variants and prepare results for comparison
EVALUATION_ARTIFACTS = {}
comparison_rows = []
fold_metric_rows = []
POOLED_PREDICTIONS = {}


def save_json(prefix, name, value):
    EVALUATION_ARTIFACTS[prefix + name] = json.dumps(
        value, indent=2
    ).encode()


def save_csv(prefix, name, frame):
    EVALUATION_ARTIFACTS[prefix + name] = (
        frame.to_csv(index=False).encode()
    )


def save_figure(prefix, name, figure):
    buffer = io.BytesIO()
    figure.savefig(
        buffer, format="png", dpi=150, bbox_inches="tight"
    )
    EVALUATION_ARTIFACTS[prefix + name] = buffer.getvalue()
    plt.close(figure)


def score_predictions(predictions):
    actual = predictions["actual"].to_numpy()
    predicted = predictions["predicted"].to_numpy()

    matrix = confusion_matrix(
        actual, predicted, labels=list(range(N_CLASSES))
    )
    support = matrix.sum(axis=1)
    present = support > 0
    recall = np.divide(
        np.diag(matrix),
        support,
        out=np.zeros(N_CLASSES, dtype=float),
        where=present,
    )

    return {
        "n_facilities": len(predictions),
        "accuracy": float(accuracy_score(actual, predicted)),
        "macro_f1": float(f1_score(
            actual, predicted,
            labels=list(range(N_CLASSES)),
            average="macro",
            zero_division=0,
        )),
        "balanced_accuracy": float(recall[present].mean()),
        "classification_report": classification_report(
            actual, predicted,
            labels=list(range(N_CLASSES)),
            target_names=list(CLASS_NAMES),
            zero_division=0,
            output_dict=True,
        ),
        "confusion_matrix": matrix.tolist(),
        "missing_true_classes": [
            CLASS_NAMES[i]
            for i in range(N_CLASSES) if not present[i]
        ],
    }


def save_prediction_results(prefix, predictions, metrics):
    save_csv(prefix, "test_predictions.csv", predictions)
    save_json(prefix, "test_metrics.json", metrics)

    per_class = pd.DataFrame([
        {"class": name, **metrics["classification_report"][name]}
        for name in CLASS_NAMES
    ])
    save_csv(prefix, "per_class_metrics.csv", per_class)

    figure, axis = plt.subplots()
    ConfusionMatrixDisplay(
        np.asarray(metrics["confusion_matrix"]),
        display_labels=CLASS_NAMES,
    ).plot(
        ax=axis, cmap="Blues", values_format="d", colorbar=False
    )
    axis.set_title("Held-out test confusion matrix")
    save_figure(prefix, "confusion_matrix.png", figure)


for variant in MODALITY_SETS:
    variant_predictions = []

    for test_fold in range(1, N_FOLDS + 1):
        model, loader, config, prefix = load_comparison_model(
            variant, test_fold
        )
        records = []

        with torch.inference_mode():
            for batch in loader:
                logits = model(
                    batch["image"].to(device),
                    batch["tabular"].to(device),
                    batch["dem"].to(device),
                    batch["dem_mask"].to(device),
                )
                assert logits.shape == (len(batch["id"]), N_CLASSES)
                assert torch.isfinite(logits).all()

                probabilities = logits.softmax(1).cpu().numpy()

                for index, facility_id in enumerate(batch["id"]):
                    records.append({
                        "id": facility_id,
                        "test_fold": test_fold,
                        "variant": variant,
                        "actual": int(batch["label"][index]),
                        "predicted": int(probabilities[index].argmax()),
                        **{
                            f"prob_{name.lower()}": float(
                                probabilities[index, i]
                            )
                            for i, name in enumerate(CLASS_NAMES)
                        },
                    })

        predictions = pd.DataFrame(records)
        expected = TABULAR_FOLDS[test_fold]["split_df"]
        expected_ids = set(
            expected.loc[expected["split"] == "test", "id"]
        )

        assert predictions["id"].is_unique
        assert set(predictions["id"]) == expected_ids

        metrics = score_predictions(predictions)
        save_prediction_results(
            prefix + "evaluation/", predictions, metrics
        )

        history = pd.read_csv(io.BytesIO(
            read_artifact(prefix + "training_history.csv")
        ))
        for metric in ("loss", "accuracy", "macro_f1"):
            figure, axis = plt.subplots()
            for split in ("train", "val"):
                axis.plot(
                    history["epoch"],
                    history[f"{split}_{metric}"],
                    label=split,
                )
            axis.axvline(
                config["best_epoch"],
                color="black", linestyle="--",
                label="selected checkpoint",
            )
            axis.set(
                xlabel="Epoch", ylabel=metric,
                title=f"{variant}, fold {test_fold}",
            )
            axis.legend()
            save_figure(
                prefix + "evaluation/",
                f"{metric}_curve.png",
                figure,
            )

        fold_metric_rows.append({
            "variant": variant,
            "test_fold": test_fold,
            **{
                name: metrics[name]
                for name in (
                    "accuracy", "macro_f1", "balanced_accuracy"
                )
            },
        })
        variant_predictions.append(predictions)
        del model, loader

    pooled = pd.concat(variant_predictions, ignore_index=True)
    assert pooled["id"].is_unique
    assert set(pooled["id"]) == set(MULTIMODAL_MANIFEST["id"])

    POOLED_PREDICTIONS[variant] = pooled
    metrics = score_predictions(pooled)

    save_prediction_results(
        MULTIMODAL_RUN_PREFIX + f"evaluation/{variant}/",
        pooled,
        metrics,
    )

    comparison_rows.append({
        "variant": variant,
        **{
            name: metrics[name]
            for name in (
                "n_facilities", "accuracy",
                "macro_f1", "balanced_accuracy",
            )
        },
    })

# Explicitly verify identical test IDs, folds, and labels across variants.
reference = POOLED_PREDICTIONS["all"].set_index("id")[
    ["test_fold", "actual"]
].sort_index()

for predictions in POOLED_PREDICTIONS.values():
    pd.testing.assert_frame_equal(
        predictions.set_index("id")[
            ["test_fold", "actual"]
        ].sort_index(),
        reference,
    )

COMPARISON = pd.DataFrame(comparison_rows)
FOLD_COMPARISON = pd.DataFrame(fold_metric_rows)
comparison_prefix = MULTIMODAL_RUN_PREFIX + "evaluation/"

save_csv(comparison_prefix, "modality_comparison.csv", COMPARISON)
save_csv(comparison_prefix, "fold_metrics.csv", FOLD_COMPARISON)

metric_names = ["accuracy", "macro_f1", "balanced_accuracy"]
fold_summary = FOLD_COMPARISON.groupby("variant")[metric_names].agg(
    ["mean", "std"]
)
fold_summary.columns = [
    f"{metric}_{statistic}"
    for metric, statistic in fold_summary.columns
]
save_csv(
    comparison_prefix,
    "fold_summary.csv",
    fold_summary.reset_index(),
)

axis = COMPARISON.set_index("variant")[metric_names].plot.bar(
    figsize=(10, 5), ylim=(0, 1), rot=15
)
axis.set(
    ylabel="Score",
    title="Modality comparison — pooled out-of-fold predictions",
)
save_figure(
    comparison_prefix, "modality_comparison.png", axis.figure
)

save_json(comparison_prefix, "evaluation_policy.json", {
    "checkpoint_selection": "lowest validation loss",
    "test_policy": "same held-out facilities for every variant",
    "pooled_policy": "one test prediction per facility per variant",
    "macro_f1": "fixed labels 0,1,2; zero_division=0",
    "balanced_accuracy": "mean recall over classes present in test data",
    "missing_classes": (
        "Report support=0; their recall is not empirically assessable."
    ),
    "comparison_table": "pooled out-of-fold metrics",
    "fold_summary": "unweighted fold mean and sample standard deviation",
})

display(COMPARISON)
display(fold_summary)
print("All variants evaluated on identical held-out facilities.")

,variant,n_facilities,accuracy,macro_f1,balanced_accuracy
0,image_only,180,0.722222,0.450043,0.459133
1,image_tabular,180,0.700000,0.409833,0.412683
2,image_dem,180,0.700000,0.400684,0.404770
3,all,180,0.677778,0.401281,0.405799


,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std
variant,,,,,,
all,0.678151,0.061761,0.381542,0.096025,0.405667,0.081366
image_dem,0.700691,0.068948,0.385045,0.089896,0.404444,0.075327
image_only,0.722445,0.036865,0.445156,0.044007,0.459000,0.051446
image_tabular,0.700215,0.043370,0.393333,0.084186,0.412444,0.061777


All variants evaluated on identical held-out facilities.


In [34]:
#upload evaluation results
content_types = {
    ".csv": "text/csv",
    ".json": "application/json",
    ".png": "image/png",
}

for key, payload in sorted(EVALUATION_ARTIFACTS.items()):
    s3_shared.upload_fileobj(
        io.BytesIO(payload),
        S3_BUCKET,
        key,
        ExtraArgs={
            "ContentType": content_types[Path(key).suffix]
        },
    )

print(
    "Section 10 results:",
    f"s3://{S3_BUCKET}/{MULTIMODAL_RUN_PREFIX}evaluation/",
)

Section 10 results: s3://atlas-acm-research-f26/artifacts/multimodal/multimodal-20261104-062610-466211/evaluation/


## 11. XAI scope — proposed
**Owner: Team — TBD**

| Component | Method |
| --- | --- |
| Image | Grad-CAM on ResNet-18 |
| Tabular | Integrated Gradients or SHAP |
| DEM | Input attribution or SmallCNN Grad-CAM |
| Modalities | Ablation comparison |

Target the final fused prediction. Start with image Grad-CAM and modality ablations; add tabular/DEM explanations afterward. Show a few correct and incorrect test predictions.

In [ ]:
# TODO: XAI methods and example visualizations.

## 12. Results and handoff
**Owner: Team — TBD**

Record checkpoint, metrics, ablations, XAI examples, artifact paths, and remaining tasks by owner.

In [ ]:
# TODO: Experiment summary and artifact paths.